# Query Distance Analysis

Quantifies how far an encoded query lives from the corpus body, in both the original 768-d embedding space and the UMAP 3D space the frontend renders.

Motivating question: in `TopicGraph3D`, the virtual query node visually lands far from the main cloud. Two things could be responsible:

1. **Semantic / embedding mismatch** — short keyword queries may genuinely sit on a different region of the encoder's output manifold than full abstracts.
2. **Rendering placement** — `CustomGraph3D.tsx:588` falls back to `(0, 0, 0)` for nodes without `x/y/z`, and `/api/query-projection` doesn't return a position. The UMAP cloud lives away from the origin, so the query renders detached regardless of how close it actually is.

This notebook separates the two effects.

## Setup

Load embeddings, paper index, and the precomputed topic graph (cluster assignments + UMAP coords). Mirrors the loaders in `notebooks/topic_graph.ipynb`.

In [10]:
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.join(os.getcwd(), ".."))
import config
from src import encoder as _enc

EMBEDDINGS_DIR = config.EMBEDDINGS_DIR
PROCESSED_DIR  = config.PROCESSED_DIR

abstracts = np.load(os.path.join(EMBEDDINGS_DIR, "abstracts.npy"))
with open(os.path.join(EMBEDDINGS_DIR, "index.json")) as f:
    index = json.load(f)
abstract_ids = index["abstracts"]

with open(os.path.join(PROCESSED_DIR, "topic_graph.json")) as f:
    topic_graph = json.load(f)

n_clusters = topic_graph["meta"]["k"]
cluster_labels = {c["id"]: c["label"] for c in topic_graph["clusters"]}

# Per-row cluster + UMAP arrays aligned with abstracts.npy ordering.
node_lookup = {n["paper_id"]: n for n in topic_graph["nodes"]}
assignments = np.array([node_lookup[pid]["cluster"] for pid in abstract_ids], dtype=np.int64)
umap_coords = np.array(
    [[node_lookup[pid]["x"], node_lookup[pid]["y"], node_lookup[pid]["z"]] for pid in abstract_ids],
    dtype=np.float64,
)

print(f"abstracts : {abstracts.shape}  ({index['encoder_model']})")
print(f"clusters  : k={n_clusters}")
print(
    f"UMAP bbox : x=[{umap_coords[:,0].min():.2f}, {umap_coords[:,0].max():.2f}], "
    f"y=[{umap_coords[:,1].min():.2f}, {umap_coords[:,1].max():.2f}], "
    f"z=[{umap_coords[:,2].min():.2f}, {umap_coords[:,2].max():.2f}]"
)
print(f"UMAP centroid (whole corpus): {umap_coords.mean(axis=0)}")

abstracts : (10000, 768)  (all-mpnet-base-v2)
clusters  : k=26
UMAP bbox : x=[2.19, 9.94], y=[4.95, 12.88], z=[3.24, 10.38]
UMAP centroid (whole corpus): [6.3764447 9.5058443 7.4817337]


## Cluster centroids (both spaces)

The topic_graph artifact stores cluster `id`, `label`, `size` but not centroid coords, so we recompute from member assignments.

- Embedding centroids are L2-renormalized so dot product with a unit-norm query is a valid cosine similarity.
- UMAP centroids are kept in raw 3D coords for Euclidean distance.

In [11]:
emb_centroids = np.zeros((n_clusters, abstracts.shape[1]), dtype=np.float32)
umap_centroids = np.zeros((n_clusters, 3), dtype=np.float64)
cluster_sizes = np.zeros(n_clusters, dtype=np.int64)
for c in range(n_clusters):
    mask = assignments == c
    cluster_sizes[c] = int(mask.sum())
    emb_centroids[c]  = abstracts[mask].mean(axis=0)
    umap_centroids[c] = umap_coords[mask].mean(axis=0)

norms = np.linalg.norm(emb_centroids, axis=1, keepdims=True)
emb_centroids_norm = emb_centroids / np.maximum(norms, 1e-12)

print(f"emb centroids   : {emb_centroids_norm.shape}")
print(f"UMAP centroids  : {umap_centroids.shape}")
print(f"cluster sizes   : min={cluster_sizes.min()}, mean={cluster_sizes.mean():.1f}, max={cluster_sizes.max()}")

emb centroids   : (26, 768)
UMAP centroids  : (26, 3)
cluster sizes   : min=194, mean=384.6, max=697


## Baseline: how far is a paper from its own cluster centroid?

Without a baseline, "the query is X away from cluster Y" is just a number. We need to know what "close" looks like for an in-distribution document so we can judge query distances against it.

- **Embedding space**: cosine distance from each paper to its own assigned cluster centroid.
- **UMAP space**: Euclidean distance from each paper's UMAP coord to its own UMAP cluster centroid.

These distributions calibrate "typical intra-cluster spread".

In [12]:
# Vectorized: gather each paper's centroid, then compute distances
own_emb_centroid  = emb_centroids_norm[assignments]
own_umap_centroid = umap_centroids[assignments]

own_emb_dist  = 1.0 - np.einsum("ij,ij->i", abstracts, own_emb_centroid)
own_umap_dist = np.linalg.norm(umap_coords - own_umap_centroid, axis=1)

def fmt_stats(name, x):
    return (
        f"{name:42}  mean={x.mean():.3f}  p50={np.median(x):.3f}  "
        f"p90={np.percentile(x, 90):.3f}  p95={np.percentile(x, 95):.3f}  max={x.max():.3f}"
    )

print(fmt_stats("Embedding cosine dist (paper→own centroid)", own_emb_dist))
print(fmt_stats("UMAP eucl. dist     (paper→own centroid)", own_umap_dist))

Embedding cosine dist (paper→own centroid)  mean=0.340  p50=0.327  p90=0.469  p95=0.524  max=0.970
UMAP eucl. dist     (paper→own centroid)    mean=1.136  p50=0.852  p90=2.257  p95=3.272  max=6.431


## Test queries

Mix of in-domain queries (covering observed cluster topics like RL, federated learning, transformer compression), short keyword queries, and one out-of-distribution control to see what "genuinely far" looks like.

In [13]:
QUERIES = [
    "transformer quantization and model compression",
    "reinforcement learning policy gradient methods",
    "federated learning with privacy guarantees",
    "weather forecasting with neural networks",
    "medical imaging diagnosis with deep learning",
    "graph neural networks",
    "attention mechanism interpretability",
    "chocolate chip cookie recipe",  # out-of-distribution control
]

print(f"Loading encoder: {index['encoder_model']}...")
model = _enc.load_model()
query_embs = _enc.encode(QUERIES, model)
query_embs = query_embs / np.maximum(np.linalg.norm(query_embs, axis=1, keepdims=True), 1e-12)
print(f"query_embs: {query_embs.shape}")

Loading encoder: all-mpnet-base-v2...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.78it/s]

query_embs: (8, 768)


## Per-query distances in both spaces

For each query we compute, in **embedding space**:

1. Cosine distance to each cluster centroid → identifies the semantically closest cluster.
2. Cosine similarity to each abstract → top-k nearest abstracts (mirrors `/api/query-projection`).

And in **UMAP 3D space**:

3. **Predicted query position** = mean of the top-k neighbors' UMAP coords (the centroid-of-neighbors fix proposed earlier).
4. Euclidean distance from that predicted position to each cluster's UMAP centroid.
5. Euclidean distance from `(0,0,0)` (the rendering bug's placement) to each cluster's UMAP centroid.

In [14]:
QUERY_K = 8  # mirrors backend QUERY_NEIGHBORS for /api/query-projection
ORIGIN  = np.zeros(3)

results = []
for q_text, q_emb in zip(QUERIES, query_embs):
    # Embedding-space distances
    cos_sims_to_abstracts = abstracts @ q_emb
    cos_dist_to_centroid  = 1.0 - emb_centroids_norm @ q_emb
    centroid_order = np.argsort(cos_dist_to_centroid)

    # Top-k neighbors → predicted UMAP position + voted cluster
    top_rows = np.argsort(-cos_sims_to_abstracts)[:QUERY_K]
    predicted_umap = umap_coords[top_rows].mean(axis=0)
    nbr_clusters = assignments[top_rows]
    voted_cluster = int(np.bincount(nbr_clusters, minlength=n_clusters).argmax())

    # UMAP distances under both placement rules
    pred_to_centroid   = np.linalg.norm(umap_centroids - predicted_umap, axis=1)
    origin_to_centroid = np.linalg.norm(umap_centroids - ORIGIN, axis=1)

    results.append({
        "query": q_text,
        "top_neighbor_cos_sim": float(cos_sims_to_abstracts[top_rows[0]]),
        "emb_closest_cluster": int(centroid_order[0]),
        "emb_closest_label":   cluster_labels[int(centroid_order[0])],
        "emb_closest_cos_dist": float(cos_dist_to_centroid[centroid_order[0]]),
        "emb_furthest_cos_dist": float(cos_dist_to_centroid[centroid_order[-1]]),
        "voted_cluster": voted_cluster,
        "voted_label":   cluster_labels[voted_cluster],
        "predicted_umap": predicted_umap,
        "pred_to_voted":     float(pred_to_centroid[voted_cluster]),
        "origin_to_voted":   float(origin_to_centroid[voted_cluster]),
        "pred_to_nearest_centroid":   float(pred_to_centroid.min()),
        "origin_to_nearest_centroid": float(origin_to_centroid.min()),
    })

### Embedding-space report

Does the query land near a cluster in the *original* (768-d) space? If yes, the encoder is doing its job; any visual distance is a rendering issue.

Compare each query's cosine distance to its closest centroid against the baseline (median paper-to-own-centroid distance).

In [15]:
import textwrap

baseline_p50 = float(np.median(own_emb_dist))
baseline_p95 = float(np.percentile(own_emb_dist, 95))

print(f"Baseline (paper→own centroid):  p50={baseline_p50:.3f}  p95={baseline_p95:.3f}")
print()
print(f"{'query':46}  {'closest cluster (label)':32}  {'cos-d':>6}  {'Δ vs p50':>9}  {'verdict':<22}")
print("-" * 130)
for r in results:
    delta = r["emb_closest_cos_dist"] - baseline_p50
    if r["emb_closest_cos_dist"] <= baseline_p50:
        verdict = "in-distribution"
    elif r["emb_closest_cos_dist"] <= baseline_p95:
        verdict = "borderline"
    else:
        verdict = "truly off-manifold"
    q_short = textwrap.shorten(r["query"], width=44, placeholder="…")
    lbl     = textwrap.shorten(r["emb_closest_label"], width=30, placeholder="…")
    print(f"{q_short:46}  {lbl:32}  {r['emb_closest_cos_dist']:>6.3f}  {delta:>+9.3f}  {verdict:<22}")

Baseline (paper→own centroid):  p50=0.327  p95=0.524

query                                           closest cluster (label)            cos-d   Δ vs p50  verdict               
----------------------------------------------------------------------------------------------------------------------------------
transformer quantization and model…             AI inference optimization          0.562     +0.235  truly off-manifold    
reinforcement learning policy gradient…         reinforcement learning             0.408     +0.081  borderline            
federated learning with privacy guarantees      federated learning                 0.233     -0.095  in-distribution       
weather forecasting with neural networks        time series forecasting            0.446     +0.119  borderline            
medical imaging diagnosis with deep learning    medical image analysis             0.398     +0.071  borderline            
graph neural networks                           graph neural networks  

### UMAP-space report — quantifying the bug

Compare two placement rules for the query node:

- **Corrected** (`predicted_umap` = mean of top-k neighbors' UMAP coords).
- **Bug** (origin `(0,0,0)`, which is what the renderer falls back to today).

Both reported as Euclidean distance to the voted-cluster centroid, scaled in units of typical intra-cluster spread (`p95(own_umap_dist)`).

In [16]:
intra_p95 = float(np.percentile(own_umap_dist, 95))

print(f"Typical intra-cluster spread (p95 paper→own UMAP centroid): {intra_p95:.3f}")
print()
hdr = (
    f"{'query':46}  {'voted cluster':30}  "
    f"{'pred d':>7}  {'bug d':>7}  {'pred (×p95)':>11}  {'bug (×p95)':>11}"
)
print(hdr)
print("-" * len(hdr))
for r in results:
    q_short = textwrap.shorten(r["query"], width=44, placeholder="…")
    lbl     = textwrap.shorten(r["voted_label"], width=28, placeholder="…")
    print(
        f"{q_short:46}  {lbl:30}  "
        f"{r['pred_to_voted']:>7.3f}  {r['origin_to_voted']:>7.3f}  "
        f"{r['pred_to_voted']/intra_p95:>10.2f}×  {r['origin_to_voted']/intra_p95:>10.2f}×"
    )

Typical intra-cluster spread (p95 paper→own UMAP centroid): 3.272

query                                           voted cluster                    pred d    bug d  pred (×p95)   bug (×p95)
--------------------------------------------------------------------------------------------------------------------------
transformer quantization and model…             AI inference optimization         0.357   13.874        0.11×        4.24×
reinforcement learning policy gradient…         reinforcement learning            0.818   14.466        0.25×        4.42×
federated learning with privacy guarantees      federated learning                0.340   13.694        0.10×        4.18×
weather forecasting with neural networks        time series forecasting           1.053   12.266        0.32×        3.75×
medical imaging diagnosis with deep learning    medical image analysis            0.507   10.765        0.16×        3.29×
graph neural networks                           graph neural networks   

### Aggregate summary

Average displacement under each rule, and the multiplier by which the bug placement exceeds typical intra-cluster spread.

In [17]:
in_dom = [r for r in results if r["query"] != "chocolate chip cookie recipe"]

pred_mean   = np.mean([r["pred_to_voted"]   for r in in_dom])
origin_mean = np.mean([r["origin_to_voted"] for r in in_dom])

print(f"In-domain queries (n={len(in_dom)}):")
print(f"  mean dist to voted-cluster centroid (corrected) : {pred_mean:.3f}  ({pred_mean/intra_p95:.2f}× typical spread)")
print(f"  mean dist to voted-cluster centroid (bug=origin): {origin_mean:.3f}  ({origin_mean/intra_p95:.2f}× typical spread)")
print(f"  bug overshoot ratio                              : {origin_mean/pred_mean:.2f}×")

In-domain queries (n=7):
  mean dist to voted-cluster centroid (corrected) : 0.789  (0.24× typical spread)
  mean dist to voted-cluster centroid (bug=origin): 13.254  (4.05× typical spread)
  bug overshoot ratio                              : 16.79×


## 3D visualization

Faint cloud = corpus colored by cluster. Green diamonds = corrected query positions (neighbor centroid). Red X = origin, where the renderer currently puts every query.

In [18]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=umap_coords[:, 0], y=umap_coords[:, 1], z=umap_coords[:, 2],
    mode="markers",
    marker=dict(size=2, color=assignments, colorscale="Viridis", opacity=0.35),
    name="papers",
    hoverinfo="skip",
))

qpos = np.array([r["predicted_umap"] for r in results])
fig.add_trace(go.Scatter3d(
    x=qpos[:, 0], y=qpos[:, 1], z=qpos[:, 2],
    mode="markers+text",
    marker=dict(size=8, color="#a3d977", symbol="diamond"),
    text=[textwrap.shorten(r["query"], width=28, placeholder="…") for r in results],
    textposition="top center",
    name="corrected query position",
))

fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers+text",
    marker=dict(size=12, color="red", symbol="x"),
    text=["origin (bug placement)"],
    textposition="top center",
    name="origin (bug)",
))

fig.update_layout(
    title="Query placement: corrected (green) vs current bug (red origin) vs paper cloud",
    scene=dict(bgcolor="#0b0f14"),
    paper_bgcolor="#0b0f14",
    font=dict(color="#e5e7eb"),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

## Preview the fix

Simulates what `TopicGraph3D` will render *after* the centroid-of-neighbors placement is applied to the virtual query node. For a single query, this cell:

1. Encodes the query and finds its top-k neighbors (mirrors `/api/query-projection`).
2. Places the query node at the mean UMAP coord of those neighbors (the fix).
3. Renders the full corpus + query node + query→neighbor edges, exactly the visual the frontend would show.

Change `PREVIEW_QUERY` to test other phrases. The query node should land *inside* the cluster of its top hits, with green edges fanning out to them.

In [19]:
PREVIEW_QUERY = "federated learning with privacy guarantees"

# 1. Encode + find top-k neighbors (mirrors /api/query-projection)
q_emb = _enc.encode([PREVIEW_QUERY], model)[0]
q_emb = q_emb / max(np.linalg.norm(q_emb), 1e-12)
cos_sims = abstracts @ q_emb
top_rows = np.argsort(-cos_sims)[:QUERY_K]

# 2. Apply the fix: place query at centroid of neighbors' UMAP coords
query_pos = umap_coords[top_rows].mean(axis=0)
voted = int(np.bincount(assignments[top_rows], minlength=n_clusters).argmax())

print(f"Query           : {PREVIEW_QUERY}")
print(f"Voted cluster   : {voted} - {cluster_labels[voted]}")
print(f"Query position  : ({query_pos[0]:.3f}, {query_pos[1]:.3f}, {query_pos[2]:.3f})")
print(f"Cluster centroid: ({umap_centroids[voted,0]:.3f}, {umap_centroids[voted,1]:.3f}, {umap_centroids[voted,2]:.3f})")
print(f"Distance        : {np.linalg.norm(query_pos - umap_centroids[voted]):.3f}  (typical intra-cluster p95: {np.percentile(own_umap_dist, 95):.3f})")
print()
print("Top neighbors:")
for r in top_rows:
    print(f"  {abstract_ids[r]}  cluster={assignments[r]:2d}  sim={cos_sims[r]:.3f}")

# 3. Render: corpus + query node + query→neighbor edges (mimics frontend)
fig = go.Figure()

# Faint corpus colored by cluster
fig.add_trace(go.Scatter3d(
    x=umap_coords[:, 0], y=umap_coords[:, 1], z=umap_coords[:, 2],
    mode="markers",
    marker=dict(size=2, color=assignments, colorscale="Viridis", opacity=0.30),
    name="papers",
    hoverinfo="skip",
))

# Highlight the top-k neighbors
fig.add_trace(go.Scatter3d(
    x=umap_coords[top_rows, 0], y=umap_coords[top_rows, 1], z=umap_coords[top_rows, 2],
    mode="markers",
    marker=dict(size=6, color="#a3d977", symbol="circle", line=dict(width=1, color="white")),
    name=f"top-{QUERY_K} neighbors",
    text=[abstract_ids[r] for r in top_rows],
    hovertemplate="%{text}<extra></extra>",
))

# Query node at fixed position
fig.add_trace(go.Scatter3d(
    x=[query_pos[0]], y=[query_pos[1]], z=[query_pos[2]],
    mode="markers+text",
    marker=dict(size=12, color="#a3d977", symbol="diamond", line=dict(width=2, color="white")),
    text=["query (after fix)"],
    textposition="top center",
    name="query node",
))

# Edges from query to each neighbor
ex, ey, ez = [], [], []
for r in top_rows:
    ex += [query_pos[0], umap_coords[r, 0], None]
    ey += [query_pos[1], umap_coords[r, 1], None]
    ez += [query_pos[2], umap_coords[r, 2], None]
fig.add_trace(go.Scatter3d(
    x=ex, y=ey, z=ez,
    mode="lines",
    line=dict(color="#a3d977", width=3),
    name="query edges",
    hoverinfo="skip",
))

# Origin marker for contrast (where the bug puts it)
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers+text",
    marker=dict(size=10, color="red", symbol="x"),
    text=["(0,0,0) — old bug placement"],
    textposition="bottom center",
    name="origin (bug)",
))

fig.update_layout(
    title=f"After fix: query \"{PREVIEW_QUERY}\" placed inside cluster {voted} ({cluster_labels[voted]})",
    scene=dict(bgcolor="#0b0f14"),
    paper_bgcolor="#0b0f14",
    font=dict(color="#e5e7eb"),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.59s/it]

Query           : federated learning with privacy guarantees
Voted cluster   : 25 - federated learning
Query position  : (8.436, 7.409, 7.607)
Cluster centroid: (8.353, 7.739, 7.606)
Distance        : 0.340  (typical intra-cluster p95: 3.272)

Top neighbors:
  2602.22633  cluster=25  sim=0.811
  2602.21844  cluster=25  sim=0.782
  2603.19040  cluster=25  sim=0.776
  2603.23472  cluster=25  sim=0.754
  2602.14244  cluster=25  sim=0.744
  2603.27986  cluster=25  sim=0.734
  2602.19945  cluster=25  sim=0.727
  2603.19617  cluster=25  sim=0.720


## Conclusion

Read the embedding-space table and the UMAP-space table together:

- If queries are at-or-below baseline cosine distance to their closest cluster centroid, the encoder is placing them sensibly inside the corpus manifold — semantically there's no "far" problem.
- If the bug-placement (origin) distance to the voted cluster is several times the typical intra-cluster spread while the corrected distance is roughly 1×, the visual gap users see in `TopicGraph3D` is an artifact of `CustomGraph3D.tsx:588`'s `node.x ?? 0` fallback, not a property of the embedding model.

The minimal fix matches what the centroid-of-neighbors `predicted_umap` does here: in `TopicGraph3D.tsx` around line 224 (where the virtual query node is constructed), assign `x/y/z` from the mean UMAP coords of the top-k neighbors before passing to `CustomGraph3D`.